# 正則化 — 係数に罰を与える

**出典**　機械学習 4 分野の教科書 — 第 2 章 1-A 回帰と正則化 ／ 04 正則化：係数が大きくなりすぎないよう罰を与える

**このノートのパス**　`ml-code/02_regression/03_regularization.ipynb`

**このノートで確かめること**
過学習した M=15 の多項式に Ridge と Lasso を掛け、**係数の最大値**と
**ゼロになった係数の個数**の違いを確かめる。alpha は交差検証で選ぶ。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, numpy` |
| 実行時間の目安 | 約 10 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

前のノートと同じデータを作り直す（**このノート単体で完結させるため**）。

In [3]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(0)
X = np.sort(rng.uniform(0, 1, 20))[:, None]
y = np.sin(2 * np.pi * X).ravel() + rng.normal(0, 0.25, 20)

In [4]:
# 過学習した M=15 の多項式に、正則化を掛けて比べる
for name, reg in [("なし",  LinearRegression()),
                  ("Ridge", Ridge(alpha=1e-3)),
                  ("Lasso", Lasso(alpha=1e-3, max_iter=100000))]:
    m = make_pipeline(PolynomialFeatures(15), StandardScaler(), reg).fit(X, y)
    coefs = m[-1].coef_
    n_zero = int(np.sum(np.abs(coefs) < 1e-8))
    print(f"{name:>6}  最大|係数|={np.abs(coefs).max():10.2f}  ゼロの個数={n_zero:2d}")

    なし  最大|係数|=3222397183.84  ゼロの個数= 1
 Ridge  最大|係数|=      5.83  ゼロの個数= 1
 Lasso  最大|係数|=      4.77  ゼロの個数=12


### alpha（λ）は交差検証で選ぶ。手で決めない。

In [5]:
ridge_cv = RidgeCV(alphas=np.logspace(-4, 2, 30), cv=5)
ridge_cv.fit(PolynomialFeatures(15).fit_transform(X), y)
print("選ばれた alpha:", ridge_cv.alpha_)

# 候補は対数スケールで置く。0.01 と 0.02 の差は 1.0 と 1.01 の差よりずっと大きい
print("候補の並び（先頭 6 個）:", np.round(np.logspace(-4, 2, 30)[:6], 5))

選ばれた alpha: 0.20433597178569418
候補の並び（先頭 6 個）: [0.0001  0.00016 0.00026 0.00042 0.00067 0.00108]


## 読み取りどころ

- **Ridge は係数を小さくするが 0 にはしない。**全特徴を使いつつ暴れを抑える。
- **Lasso は一部の係数をちょうど 0 にする。**＝特徴選択が同時に走る。
  出力のうち「ゼロの個数」が Lasso だけ大きくなるのはこのため。
- alpha は**対数スケール**で候補を置く（`np.logspace`）。正則化の強さは桁で効くので、等間隔に並べると大半が同じ強さになる。

この「桁で効く量は対数一様で引く」という原則は、**1-D のハイパーパラメータ探索でも同じように現れる**。